# Homework GraphFrames Part 2 Consumer

In [1]:
import json
import sys
import time
from collections import defaultdict

sys.path.insert(0, "/home/hadoop/homework/spark-graphframe")

from kafka import KafkaConsumer, TopicPartition
from pyspark.sql import functions as F

from spark_graphframe_homework import (
    BOOTSTRAP_SERVERS,
    TOPIC_NAME,
    build_realtime_route_graph,
    create_kafka_topic,
    create_spark_session,
    distance_between_stations_km,
    load_station_lookup,
    read_latest_run_id,
    route_count_rows,
)


In [2]:
create_kafka_topic(TOPIC_NAME)

RUN_ID = read_latest_run_id()
MAX_MESSAGES = 250
spark = create_spark_session("graphframes-homework-consumer")
station_lookup = load_station_lookup()

{"run_id": RUN_ID, "max_messages": MAX_MESSAGES, "known_stations": len(station_lookup)}


26/03/17 08:31:31 WARN Utils: Your hostname, bigdata resolves to a loopback address: 127.0.1.1; using 10.3.134.62 instead (on interface ens3)
26/03/17 08:31:31 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
26/03/17 08:31:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


{'run_id': 'trip-run-20260317T082324Z',
 'max_messages': 250,
 'known_stations': 70}

In [3]:
def consume_trip_stream(run_id=RUN_ID, max_messages=MAX_MESSAGES, consumer_timeout_ms=5000):
    consumer = KafkaConsumer(
        bootstrap_servers=[BOOTSTRAP_SERVERS],
        enable_auto_commit=False,
        consumer_timeout_ms=consumer_timeout_ms,
        value_deserializer=lambda raw: json.loads(raw.decode("utf-8")),
    )

    topic_partitions = [TopicPartition(TOPIC_NAME, 0)]
    consumer.assign(topic_partitions)
    consumer.seek_to_beginning(*topic_partitions)

    route_counts = defaultdict(int)
    bike_distance_km = defaultdict(float)
    alerted_bikes = set()
    alerts = []
    processed = 0

    try:
        for record in consumer:
            message = record.value
            if run_id and message.get("run_id") != run_id:
                continue

            src = int(message["src"])
            dst = int(message["dst"])
            bike_id = int(message["bike_id"])

            if src not in station_lookup or dst not in station_lookup:
                continue

            distance_km = distance_between_stations_km(station_lookup, src, dst)
            route_counts[(src, dst)] += 1
            bike_distance_km[bike_id] += distance_km

            if bike_distance_km[bike_id] > 15 and bike_id not in alerted_bikes:
                alerted_bikes.add(bike_id)
                alerts.append(
                    {
                        "bike_id": bike_id,
                        "total_distance_km": round(bike_distance_km[bike_id], 3),
                        "src_name": station_lookup[src]["name"],
                        "dst_name": station_lookup[dst]["name"],
                    }
                )

            processed += 1
            if processed >= max_messages:
                break
    finally:
        consumer.close()

    route_graph = build_realtime_route_graph(spark, dict(route_counts), station_lookup)
    route_rows = route_count_rows(dict(route_counts), station_lookup)

    route_schema = "src long, src_name string, dst long, dst_name string, trip_count long, distance_km double"
    alert_schema = "bike_id long, total_distance_km double, src_name string, dst_name string"

    route_counts_df = (
        spark.createDataFrame(route_rows)
        if route_rows
        else spark.createDataFrame([], schema=route_schema)
    )
    alerts_df = (
        spark.createDataFrame(alerts)
        if alerts
        else spark.createDataFrame([], schema=alert_schema)
    )

    return {
        "processed_messages": processed,
        "route_graph": route_graph,
        "route_counts_df": route_counts_df,
        "alerts_df": alerts_df,
    }


In [4]:
consumer_result = consume_trip_stream()
consumer_result["processed_messages"]


/home/hadoop/homework/.venv/lib/python3.10/site-packages/pyspark/sql/dataframe.py:148: UserWarning: DataFrame.sql_ctx is an internal property, and will be removed in future releases. Use DataFrame.sparkSession instead.
  warnings.warn(


250

In [5]:
if consumer_result["processed_messages"] == 0:
    print("No matching Kafka messages were consumed. Run producer.ipynb first.")
else:
    trip_summary_df = (
        consumer_result["route_graph"]
        .triplets
        .select(
            F.col("edge.trip_count").alias("trip_count"),
            F.col("src.name").alias("from_station"),
            F.col("dst.name").alias("to_station"),
            F.col("edge.distance_km").alias("distance_km"),
        )
        .orderBy(F.desc("trip_count"), F.desc("distance_km"))
    )
    trip_summary_df.show(10, truncate=False)


/home/hadoop/homework/.venv/lib/python3.10/site-packages/pyspark/sql/dataframe.py:127: UserWarning: DataFrame constructor is internal. Do not directly use it.
  warnings.warn("DataFrame constructor is internal. Do not directly use it.")


+----------+---------------------------------------------+----------------------------------------+-----------+
|trip_count|from_station                                 |to_station                              |distance_km|
+----------+---------------------------------------------+----------------------------------------+-----------+
|6         |San Francisco Caltrain 2 (330 Townsend)      |Townsend at 7th                         |0.886      |
|5         |5th at Howard                                |San Francisco Caltrain 2 (330 Townsend) |1.024      |
|4         |Steuart at Market                            |San Francisco Caltrain (Townsend at 4th)|1.95       |
|4         |Embarcadero at Sansome                       |Steuart at Market                       |1.413      |
|4         |Mountain View Caltrain Station               |Castro Street and El Camino Real        |1.119      |
|3         |Embarcadero at Folsom                        |Embarcadero at Sansome                  |1.827

In [6]:
consumer_result["alerts_df"].orderBy(F.desc("total_distance_km"), F.asc("bike_id")).show(20, truncate=False)


+-------+-----------------+--------+--------+
|bike_id|total_distance_km|src_name|dst_name|
+-------+-----------------+--------+--------+
+-------+-----------------+--------+--------+

